In [33]:
from article_utils.util import read_jsonl, normalize_and_expand_column, dump_jsonl
from civic.process_revisions import normalize_evidence_direction, normalize_significance, normalize_variant_origin, resolve_revision_links

LAST_BUILD = '2026-03-03'

df = read_jsonl('../results/civic/civic-dump.rev.doc_meta.drug_info.pheno_info.gene_info.celline_info.disease_info.jsonl.gz')
df['civic.id'] = df['id']
df = df[df['ground_truth'] == True]
print(df.columns.tolist())
df = df.drop_duplicates(['id', 'revision_state'])
df['document.pmid'] = df['source'].str['citationId']

# df['events'] = df['events'].str['nodes']
df['rev'] = df['revisions.nodes']

# df['evidenceDirection'] = df.evidenceDirection.apply(normalize_evidence_direction)
# df['significance'] = df.significance.apply(normalize_significance)
# df['variantOrigin'] = df.variantOrigin.apply(normalize_variant_origin)

# df['therapyInteractionType'] = df['therapyInteractionType'].fillna('SUBSTITUTES').str.upper()

df.head()

['auto_accepted', 'corrections', 'description', 'disease', 'document', 'events', 'evidenceDirection', 'evidenceLevel', 'evidenceRating', 'evidenceType', 'flagged', 'ground_truth', 'id', 'lastUpdatedAt', 'molecularProfile', 'phenotypes', 'revision_note', 'revision_state', 'revisions.nodes', 'significance', 'source', 'status', 'submittedAt', 'therapies', 'therapyInteractionType', 'variantOrigin', 'civic.id']


,auto_accepted,corrections,description,disease,document,events,evidenceDirection,evidenceLevel,evidenceRating,evidenceType,...,significance,source,status,submittedAt,therapies,therapyInteractionType,variantOrigin,civic.id,document.pmid,rev
0,NaN,{},EML4-ALK fusion positive lung adenocarcinoma c...,"{'diseaseAliases': ['Adenocarcinoma', 'Adenoca...","{'content_source': 'pmc', 'doi': '10.1016/j.eb...","{'nodes': [{'action': 'REVISION_ACCEPTED', 'cr...",DOES_NOT_SUPPORT,D,3.0,PREDICTIVE,...,resistance,"{'citationId': '26870817', 'name': 'PubMed: Ka...",ACCEPTED,2021-12-31T03:21:40Z,"[{'id': 17, 'name': 'Alectinib', 'ncitId': 'C1...",SUBSTITUTES,SOMATIC,10016,26870817,[{'creationActivity.createdAt': '2025-11-25T20...
2,NaN,{},Seven T cell acute lymphoblastic leukemia (T-A...,NaN,"{'content_source': 'pmc', 'doi': '10.1084/jem....","{'nodes': [{'action': 'REVISION_ACCEPTED', 'cr...",SUPPORTS,D,3.0,FUNCTIONAL,...,gain of function,"{'citationId': '17646409', 'name': 'PubMed: O'...",SUBMITTED,2022-01-05T23:35:11Z,[],SUBSTITUTES,N/A,10044,17646409,[{'creationActivity.createdAt': '2025-12-11T18...
4,NaN,{},Seven T cell acute lymphoblastic leukemia (T-A...,NaN,"{'content_source': 'pmc', 'doi': '10.1084/jem....","{'nodes': [{'action': 'REVISION_ACCEPTED', 'cr...",SUPPORTS,D,3.0,FUNCTIONAL,...,gain of function,"{'citationId': '17646409', 'name': 'PubMed: O'...",SUBMITTED,2022-01-05T23:35:53Z,[],SUBSTITUTES,N/A,10045,17646409,[{'creationActivity.createdAt': '2025-12-11T18...
6,NaN,{},Seven T cell acute lymphoblastic leukemia (T-A...,NaN,"{'content_source': 'pmc', 'doi': '10.1084/jem....","{'nodes': [{'action': 'REVISION_ACCEPTED', 'cr...",DOES_NOT_SUPPORT,D,3.0,FUNCTIONAL,...,gain of function,"{'citationId': '17646409', 'name': 'PubMed: O'...",SUBMITTED,2022-01-05T23:36:39Z,[],SUBSTITUTES,N/A,10046,17646409,[{'creationActivity.createdAt': '2025-12-11T18...
8,NaN,{},Seven T cell acute lymphoblastic leukemia (T-A...,NaN,"{'content_source': 'pmc', 'doi': '10.1084/jem....","{'nodes': [{'action': 'REVISION_ACCEPTED', 'cr...",DOES_NOT_SUPPORT,D,3.0,FUNCTIONAL,...,gain of function,"{'citationId': '17646409', 'name': 'PubMed: O'...",SUBMITTED,2022-01-05T23:37:12Z,[],SUBSTITUTES,N/A,10047,17646409,[{'creationActivity.createdAt': '2025-12-11T18...


In [34]:
event_df = df.copy()
event_df['events'] = event_df['events'].str['nodes']
event_df = event_df.explode('events')
print(event_df['events'].tolist()[0])
event_df['event_action'] = event_df.events.str['action']
event_df['event_created'] = event_df.events.str['createdAt']
event_df = event_df[event_df.event_created >= LAST_BUILD ]
event_df = event_df[event_df.event_action.isin({'SUBMITTED'})]
new_eids = set(event_df['id'].tolist())
print('new EIDs', len(new_eids))
event_df.event_action.value_counts()

{'action': 'REVISION_ACCEPTED', 'createdAt': '2025-11-25T20:38:53Z', 'originatingUser': {'displayName': 'CamGrisdale', 'role': 'EDITOR'}}
new EIDs 123


event_action
SUBMITTED    123
Name: count, dtype: int64

In [35]:
def pending_editor_revisions(revisions):
    for rev in revisions:
        if rev['status'] == 'NEW' and rev['creationActivity.user.role'] in {'ADMIN', 'EDITOR'} and rev['fieldName'] not in {'description', 'rating', 'evidence_level'}:
            return True
    return False

df['pending_editor_revisions'] = df.rev.map(pending_editor_revisions)
df = df[df.pending_editor_revisions != True]
df['ground_truth'] = True

rev_df = df.explode('rev').reset_index(drop=True)
rev_df = rev_df[~rev_df.rev.isna()]
rev_df = normalize_and_expand_column(rev_df, 'rev')
rev_df = rev_df[rev_df['rev.creationActivity.createdAt'] > LAST_BUILD]
rev_df = rev_df[~rev_df['rev.fieldName'].isin({'description', 'rating', 'evidence_level'})]
rev_df = rev_df[rev_df['rev.status'].isin({'ACCEPTED', 'REJECTED'})]
rev_ids = set(rev_df['rev.id'].tolist())
print('revisions since', len(rev_ids))
resolve_revision_links(rev_df)

rev_df = (
    rev_df.groupby(['id'], dropna=False)
    .apply(
        lambda s: s[[c for c in rev_df.columns if c.startswith('rev.')]]
        .rename(columns={c: c[len('rev.') :] for c in rev_df.columns if c.startswith('rev.')})
        .to_dict('records')
    )
    .reset_index(name='revisions.nodes')
    .merge(df.drop(columns=['revisions.nodes']), on='id')
)
rev_df['ground_truth'] = True
rev_df.head()

revisions since 108


/var/tmp/ipykernel_1005755/4142412849.py:23: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(


,id,revisions.nodes,auto_accepted,corrections,description,disease,document,events,evidenceDirection,evidenceLevel,...,source,status,submittedAt,therapies,therapyInteractionType,variantOrigin,civic.id,document.pmid,rev,pending_editor_revisions
0,10339,[{'creationActivity.createdAt': '2026-05-07T03...,True,{},DDX41 p.G530D (NM_016222.2) was detected in th...,"{'diseaseAliases': ['AML', 'AML - Acute Myeloi...","{'content_source': 'pubmed', 'doi': '10.1182/b...","{'nodes': [{'action': 'REVISION_SUGGESTED', 'c...",SUPPORTS,B,...,"{'citationId': '26712909', 'name': 'PubMed: Le...",SUBMITTED,2022-05-24T04:00:49Z,[],SUBSTITUTES,RARE_GERMLINE,10339,26712909,[{'creationActivity.createdAt': '2026-05-07T03...,False
1,10344,[{'creationActivity.createdAt': '2026-05-07T03...,True,{},Patient No. 9 is a 76yo male with karyotypical...,"{'diseaseAliases': ['Myeloid Neoplasm', 'myelo...","{'content_source': 'pmc_author_fulltext', 'doi...","{'nodes': [{'action': 'REVISION_SUGGESTED', 'c...",SUPPORTS,B,...,"{'citationId': '32307695', 'name': 'PubMed: Qu...",SUBMITTED,2022-05-24T05:14:43Z,[],SUBSTITUTES,SOMATIC,10344,32307695,[{'creationActivity.createdAt': '2026-05-07T03...,False
2,10346,[{'creationActivity.createdAt': '2026-05-07T03...,True,{},A 60yo American male (Patient #30) with CN-AML...,"{'diseaseAliases': ['AML', 'AML - Acute Myeloi...","{'content_source': 'pmc', 'doi': '10.3389/fonc...","{'nodes': [{'action': 'COMMENTED', 'createdAt'...",SUPPORTS,C,...,"{'citationId': '33585199', 'name': 'PubMed: Ba...",SUBMITTED,2022-05-25T01:08:14Z,[],SUBSTITUTES,SOMATIC,10346,33585199,[{'creationActivity.createdAt': '2026-05-07T03...,False
3,10600,[{'creationActivity.createdAt': '2026-04-28T16...,NaN,{},"In this study, 91 patients with diffuse intrin...",{'diseaseAliases': ['Diffuse Intrinsic Pontine...,"{'content_source': 'pmc', 'doi': '10.1007/s004...","{'nodes': [{'action': 'REVISION_SUGGESTED', 'c...",SUPPORTS,B,...,"{'citationId': '26399631', 'name': 'PubMed: Ca...",SUBMITTED,2022-07-21T20:51:08Z,[],SUBSTITUTES,SOMATIC,10600,26399631,[{'creationActivity.createdAt': '2023-11-28T19...,False
4,10602,[{'creationActivity.createdAt': '2026-04-28T16...,NaN,{},Diffuse intrinsic pontine glioma (DIPG) patien...,{'diseaseAliases': ['Diffuse Intrinsic Pontine...,"{'content_source': 'pmc', 'doi': '10.1007/s004...","{'nodes': [{'action': 'REVISION_SUGGESTED', 'c...",SUPPORTS,D,...,"{'citationId': '26399631', 'name': 'PubMed: Ca...",SUBMITTED,2022-07-21T20:53:47Z,[],SUBSTITUTES,SOMATIC,10602,26399631,[{'creationActivity.createdAt': '2023-11-28T19...,False


In [36]:
from copy import copy
import pandas as pd

from civic.process_revisions import resolve_revision_links, apply_revision, revert_revision, generate_corrections
from nli.util import build_civic_claim, concat_text


def create_new_revision_states(row):
    if not hasattr(row['revisions.nodes'], '__iter__'):
        return None
    revisions = [
        r for r in row['revisions.nodes'] if
            r['creationActivity.createdAt'] > LAST_BUILD
            and r['status'] in {'REJECTED', 'ACCEPTED'}
            and r['fieldName'] not in {'description', 'rating', 'evidence_level'}
    ]
    if len(revisions) == 0:
        return None


    rejected = [r for r in revisions if r['status'] == 'REJECTED']
    accepted = [r for r in revisions if r['status'] == 'ACCEPTED']

    new_states = []
    if rejected:
        new_row = copy(dict(row))
        for rev in rejected:
            apply_revision(new_row, rev)
        new_row['revisions'] = []
        new_row['revision_state'] = ''.join(
            sorted({f"+RID{int(r['id'])}" for r in sorted(rejected, key=lambda x: x['id'])})
        )
        new_row['revision_fields'] = ''.join(
                    sorted({r['fieldName'] for r in sorted(rejected, key=lambda x: x['id'])})
                )
        new_row['ground_truth'] = False
        new_row['corrections'] = generate_corrections(new_row, row)
        new_states.append(new_row)
    if accepted:
        new_row = copy(dict(row))
        for rev in accepted:
            revert_revision(new_row, rev)
        new_row['revisions'] = []
        new_row['revision_state'] = ''.join(
            sorted({f"-RID{int(r['id'])}" for r in sorted(accepted, key=lambda x: x['id'])})
        )
        new_row['revision_fields'] = ''.join(
            sorted({r['fieldName'] for r in sorted(accepted, key=lambda x: x['id'])})
        )
        new_row['corrections'] = generate_corrections(new_row, row)
        new_row['ground_truth'] = False
        new_states.append(new_row)
    return new_states



rev_df['new_states'] = rev_df.apply(create_new_revision_states, axis=1)
rev_df = rev_df.explode('new_states')
rev_df = rev_df[~rev_df.new_states.isna()]
rev_df  = pd.DataFrame.from_records(rev_df['new_states'].tolist())
rev_df = rev_df[rev_df.corrections.str.len() > 0]
print(rev_df.id.nunique())
rev_df['gold_label_name'] = 'REVISIONS'
rev_df= pd.json_normalize(rev_df.to_dict('records'))

rev_df.head()

53


,id,revisions.nodes,auto_accepted,description,evidenceDirection,evidenceLevel,evidenceRating,evidenceType,flagged,ground_truth,...,source.sourceType,corrections.evidenceType,corrections.significance,corrections.therapies,corrections.therapyInteractionType,corrections.disease,disease.displayName,disease,corrections.evidenceDirection,corrections.variantOrigin
0,10339,[{'creationActivity.createdAt': '2026-05-07T03...,True,DDX41 p.G530D (NM_016222.2) was detected in th...,SUPPORTS,B,4.0,PREDISPOSING,False,False,...,PUBMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,10344,[{'creationActivity.createdAt': '2026-05-07T03...,True,Patient No. 9 is a 76yo male with karyotypical...,SUPPORTS,B,4.0,ONCOGENIC,False,False,...,PUBMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,10346,[{'creationActivity.createdAt': '2026-05-07T03...,True,A 60yo American male (Patient #30) with CN-AML...,SUPPORTS,C,5.0,ONCOGENIC,False,False,...,PUBMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,10600,[{'creationActivity.createdAt': '2026-04-28T16...,NaN,"In this study, 91 patients with diffuse intrin...",SUPPORTS,B,4.0,DIAGNOSTIC,False,False,...,PUBMED,"[DIAGNOSTIC, ONCOGENIC]","[positive, oncogenicity]",NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10605,[{'creationActivity.createdAt': '2026-04-28T16...,NaN,This retrospective study analyzed 91 biopsy-co...,SUPPORTS,B,3.0,DIAGNOSTIC,False,False,...,PUBMED,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
def _build_claim(row):
    try:
        return build_civic_claim(row, '')
    except KeyError as err:
        pass

rev_df['claim'] = rev_df.apply(_build_claim, axis=1)

print(rev_df.claim.isna().sum(), rev_df.shape[0])
rev_df = rev_df[~rev_df.claim.isna()]
rev_df['claim.flat'] = rev_df.claim.apply(lambda c: concat_text([c]))
rev_df.head()

5 54


,id,revisions.nodes,auto_accepted,description,evidenceDirection,evidenceLevel,evidenceRating,evidenceType,flagged,ground_truth,...,corrections.significance,corrections.therapies,corrections.therapyInteractionType,corrections.disease,disease.displayName,disease,corrections.evidenceDirection,corrections.variantOrigin,claim,claim.flat
0,10339,[{'creationActivity.createdAt': '2026-05-07T03...,True,DDX41 p.G530D (NM_016222.2) was detected in th...,SUPPORTS,B,4.0,PREDISPOSING,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'DDX41 G530D...,"{\n ""disease"": ""Acute Myeloid Leukemia"",\n ""..."
1,10344,[{'creationActivity.createdAt': '2026-05-07T03...,True,Patient No. 9 is a 76yo male with karyotypical...,SUPPORTS,B,4.0,ONCOGENIC,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'DDX41 G530D...,"{\n ""disease"": ""Myeloid Neoplasm"",\n ""eviden..."
2,10346,[{'creationActivity.createdAt': '2026-05-07T03...,True,A 60yo American male (Patient #30) with CN-AML...,SUPPORTS,C,5.0,ONCOGENIC,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'DDX41 G530D...,"{\n ""disease"": ""Acute Myeloid Leukemia"",\n ""..."
3,10600,[{'creationActivity.createdAt': '2026-04-28T16...,NaN,"In this study, 91 patients with diffuse intrin...",SUPPORTS,B,4.0,DIAGNOSTIC,False,False,...,"[positive, oncogenicity]",NaN,NaN,NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'H3-3A K28M'...,"{\n ""disease"": ""Diffuse Midline Glioma, H3 K2..."
5,10785,[{'creationActivity.createdAt': '2026-05-14T14...,True,An updated analysis of the phase 2 ROAR basket...,SUPPORTS,A,4.0,PREDICTIVE,False,False,...,NaN,"[{Dabrafenib/Trametinib Regimen}, {Dabrafenib,...","[SUBSTITUTES, COMBINATION]",NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'BRAF V600E'...,"{\n ""disease"": ""Anaplastic Thyroid Carcinoma""..."


In [38]:
claims_df = read_jsonl('../results/civic/claims.jsonl.gz')
claims_df = claims_df[claims_df['civic.ground_truth'] == True]
claims_df = claims_df.explode('civic.id')
claims_df = claims_df.rename(columns={'civic.id': 'id'})



recent_claims_df = df.merge(claims_df[['id', 'claim.flat']])
recent_claims_df = recent_claims_df[recent_claims_df['id'].isin(new_eids)]
print('submitted claims', recent_claims_df.shape[0])
recent_claims_df['gold_label_name'] = 'SUPPORTS'
recent_claims_df.head()

submitted claims 122


,auto_accepted,corrections,description,disease,document,events,evidenceDirection,evidenceLevel,evidenceRating,evidenceType,...,submittedAt,therapies,therapyInteractionType,variantOrigin,civic.id,document.pmid,rev,pending_editor_revisions,claim.flat,gold_label_name
596,NaN,{},"In this study, the authors analyzed 14 cases o...",{'diseaseAliases': ['Angiomatoid Fibrous Histi...,"{'content_source': 'pubmed', 'doi': '10.1158/1...","{'nodes': [{'action': 'REVISION_ACCEPTED', 'cr...",SUPPORTS,B,3.0,DIAGNOSTIC,...,2026-03-10T23:22:26Z,[],SUBSTITUTES,SOMATIC,12852,18094413,[{'creationActivity.createdAt': '2026-03-24T23...,False,"{\n ""disease"": ""Angiomatoid Fibrous Histiocyt...",SUPPORTS
597,NaN,{},"In this study, the authors report a 34‑year-ol...",{'diseaseAliases': ['Angiomatoid Fibrous Histi...,"{'content_source': 'pmc', 'doi': '10.3389/fonc...","{'nodes': [{'action': 'REVISION_ACCEPTED', 'cr...",SUPPORTS,C,3.0,DIAGNOSTIC,...,2026-03-10T23:25:29Z,[],SUBSTITUTES,SOMATIC,12853,39845329,[{'creationActivity.createdAt': '2026-03-24T23...,False,"{\n ""disease"": ""Angiomatoid Fibrous Histiocyt...",SUPPORTS
598,NaN,{},The EWSR1::CREB1 fusion associated with t(2;22...,{'diseaseAliases': ['Connective And Soft Tissu...,"{'content_source': 'pmc', 'doi': '10.3389/fonc...","{'nodes': [{'action': 'REVISION_ACCEPTED', 'cr...",SUPPORTS,C,1.0,PROGNOSTIC,...,2026-03-10T23:27:37Z,[],SUBSTITUTES,SOMATIC,12854,39845329,[{'creationActivity.createdAt': '2026-05-16T20...,False,"{\n ""disease"": ""Sarcoma"",\n ""evidenceDirecti...",SUPPORTS
599,NaN,{},"In this study, the authors analyzed 9 cases of...",{'diseaseAliases': ['Angiomatoid Fibrous Histi...,"{'content_source': 'pubmed', 'doi': '10.1002/g...","{'nodes': [{'action': 'REVISION_ACCEPTED', 'cr...",SUPPORTS,B,3.0,DIAGNOSTIC,...,2026-03-10T23:36:38Z,[],SUBSTITUTES,SOMATIC,12855,17724745,[{'creationActivity.createdAt': '2026-03-24T23...,False,"{\n ""disease"": ""Angiomatoid Fibrous Histiocyt...",SUPPORTS
600,True,{},A pBABE expression vector with wild-type FLT3 ...,"{'diseaseAliases': ['AML', 'AML - Acute Myeloi...","{'content_source': 'pubmed', 'doi': '10.1182/b...","{'nodes': [{'action': 'REVISION_SUGGESTED', 'c...",SUPPORTS,D,4.0,ONCOGENIC,...,2026-03-18T00:32:34Z,[],SUBSTITUTES,SOMATIC,12858,33914060,[{'creationActivity.createdAt': '2026-05-30T04...,False,"{\n ""disease"": ""Acute Myeloid Leukemia"",\n ""...",SUPPORTS


In [39]:
rev_df = rev_df.merge(claims_df[['id', 'claim.flat']].rename(columns={'claim.flat': 'revised_claim.flat'}))
print((rev_df['claim.flat'] == rev_df['revised_claim.flat']).sum())
rev_df.head()

0


,id,revisions.nodes,auto_accepted,description,evidenceDirection,evidenceLevel,evidenceRating,evidenceType,flagged,ground_truth,...,corrections.therapies,corrections.therapyInteractionType,corrections.disease,disease.displayName,disease,corrections.evidenceDirection,corrections.variantOrigin,claim,claim.flat,revised_claim.flat
0,10339,[{'creationActivity.createdAt': '2026-05-07T03...,True,DDX41 p.G530D (NM_016222.2) was detected in th...,SUPPORTS,B,4.0,PREDISPOSING,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'DDX41 G530D...,"{\n ""disease"": ""Acute Myeloid Leukemia"",\n ""...","{\n ""disease"": ""Acute Myeloid Leukemia"",\n ""..."
1,10344,[{'creationActivity.createdAt': '2026-05-07T03...,True,Patient No. 9 is a 76yo male with karyotypical...,SUPPORTS,B,4.0,ONCOGENIC,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'DDX41 G530D...,"{\n ""disease"": ""Myeloid Neoplasm"",\n ""eviden...","{\n ""disease"": ""Myeloid Neoplasm"",\n ""eviden..."
2,10346,[{'creationActivity.createdAt': '2026-05-07T03...,True,A 60yo American male (Patient #30) with CN-AML...,SUPPORTS,C,5.0,ONCOGENIC,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'DDX41 G530D...,"{\n ""disease"": ""Acute Myeloid Leukemia"",\n ""...","{\n ""disease"": ""Acute Myeloid Leukemia"",\n ""..."
3,10600,[{'creationActivity.createdAt': '2026-04-28T16...,NaN,"In this study, 91 patients with diffuse intrin...",SUPPORTS,B,4.0,DIAGNOSTIC,False,False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'H3-3A K28M'...,"{\n ""disease"": ""Diffuse Midline Glioma, H3 K2...","{\n ""disease"": ""Diffuse Midline Glioma, H3 K2..."
4,10785,[{'creationActivity.createdAt': '2026-05-14T14...,True,An updated analysis of the phase 2 ROAR basket...,SUPPORTS,A,4.0,PREDICTIVE,False,False,...,"[{Dabrafenib/Trametinib Regimen}, {Dabrafenib,...","[SUBSTITUTES, COMBINATION]",NaN,NaN,NaN,NaN,NaN,Text(content={'molecularProfile': 'BRAF V600E'...,"{\n ""disease"": ""Anaplastic Thyroid Carcinoma""...","{\n ""disease"": ""Anaplastic Thyroid Carcinoma""..."


In [ ]:
import uuid

output_df = pd.concat([rev_df, recent_claims_df])
print(output_df.shape)
output_df['id'] = output_df['id'].astype(str) + output_df['revision_state'].fillna('')
output_df['display_id'] = df['id'].apply(lambda _: str(uuid.uuid4()))
# dump_jsonl(output_df, '../builds/civicfactnew-2026.06.09/data.jsonl.gz')

(171, 61)
